# Análisis Territorial



## Objetivo




## Marco Conceptual

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from matplotlib.patches import Patch
from scipy import stats
from IPython.display import display

from src.config import PATHS, VIZ
from src.viz import set_style, save_fig, annotate_source
from src.territory import (
 compute_departmental_indicators,
 compute_regional_indicators,
 compute_territorial_disparities,
 REGION_COLORS,
 REGION_MAP
)

pd.set_option("display.max_rows", 40)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.2f}".format)

print(f" ROOT: {ROOT}")

## 1. Carga de Datasets Analíticos

In [ ]:
df_h = pd.read_parquet(PATHS["analytical"] / "df_hogar_analytic.parquet")
df_p = pd.read_parquet(PATHS["analytical"] / "df_persona_analytic.parquet")

print(f"Dataset Hogar: {df_h.shape[0]:,} filas x {df_h.shape[1]} columnas")
print(f"Dataset Persona: {df_p.shape[0]:,} filas x {df_p.shape[1]} columnas")
print(f"\nDepartamentos únicos en Hogar: {df_h['codigo_dpto'].nunique()}")
print(f"Departamentos únicos en Persona: {df_p['codigo_dpto'].nunique()}")
print(f"\nPoblación expandida (hogares): {df_h['fex_c'].sum():,.0f}")
print(f"Población expandida (personas): {df_p['fex_c'].sum():,.0f}")

## 2. Indicadores Ponderados por Departamento




- `pct_computador`: Proporción de hogares con computador/portátil/tablet.
- `pct_internet_fijo`: Proporción con acceso a banda ancha fija.
- `pct_internet_movil`: Proporción con acceso a Internet móvil.
- `pct_alta_dificultad`: Proporción con dificultad económica alta o muy alta.
- `pct_educ_superior_persona`: Personas con educación superior completa.

In [ ]:
df_dpto = compute_departmental_indicators(df_h, df_p)

print(f"Tabla departamental: {df_dpto.shape[0]} departamentos x {df_dpto.shape[1]} indicadores\n")

cols_show = [
 "departamento", "region", "n_hogares", "hogares_expandidos",
 "pct_computador", "pct_internet_fijo", "pct_internet_movil",
 "pct_alta_dificultad", "pct_uso_internet_persona", "pct_educ_superior_persona"
]
display(df_dpto[cols_show].to_string(index=False))

## 3. Métricas de Disparidad Territorial

In [ ]:
metrics = compute_territorial_disparities(df_dpto)

print("="*65)
print(" MÉTRICAS DE DISPARIDAD TERRITORIAL — BRECHA DIGITAL (ECV 2024) ")
print("="*65)

print("\n[COMPUTADOR EN HOGAR]")
top_dpto = df_dpto.loc[df_dpto['pct_computador'].idxmax(), 'departamento']
bot_dpto = df_dpto.loc[df_dpto['pct_computador'].idxmin(), 'departamento']
print(f" Máximo: {metrics['computador_max']:.1f}% ({top_dpto})")
print(f" Mínimo: {metrics['computador_min']:.1f}% ({bot_dpto})")
print(f" Brecha Max-Min: {metrics['computador_brecha_pp']:.1f} puntos porcentuales")
print(f" Razón Max/Min: {metrics['computador_ratio_max_min']:.1f}x")
print(f" IQR interdepartamental: {metrics['computador_iqr']:.1f} pp")

print("\n[INTERNET FIJO EN HOGAR]")
top_f = df_dpto.loc[df_dpto['pct_internet_fijo'].idxmax(), 'departamento']
bot_f = df_dpto.loc[df_dpto['pct_internet_fijo'].idxmin(), 'departamento']
print(f" Máximo: {metrics['fijo_max']:.1f}% ({top_f})")
print(f" Mínimo: {metrics['fijo_min']:.1f}% ({bot_f})")
print(f" Brecha Max-Min: {metrics['fijo_brecha_pp']:.1f} puntos porcentuales")

print("\n[USO DE INTERNET — PERSONAS]")
top_u = df_dpto.loc[df_dpto['pct_uso_internet_persona'].idxmax(), 'departamento']
bot_u = df_dpto.loc[df_dpto['pct_uso_internet_persona'].idxmin(), 'departamento']
print(f" Máximo: {metrics['uso_max']:.1f}% ({top_u})")
print(f" Mínimo: {metrics['uso_min']:.1f}% ({bot_u})")
print(f" Brecha Max-Min: {metrics['uso_brecha_pp']:.1f} puntos porcentuales")

# Referencia nacional ponderada
nat_comp = (df_h['tiene_computador'] * df_h['fex_c']).sum() / df_h['fex_c'].sum() * 100
nat_fijo = (df_h['tiene_internet_fijo'] * df_h['fex_c']).sum() / df_h['fex_c'].sum() * 100
nat_uso = (df_p['usa_internet'] * df_p['fex_c']).sum() / df_p['fex_c'].sum() * 100
print("\n[REFERENCIA NACIONAL PONDERADA]")
print(f" Tenencia computador: {nat_comp:.1f}%")
print(f" Internet fijo hogar: {nat_fijo:.1f}%")
print(f" Uso Internet persona: {nat_uso:.1f}%")
print("="*65)

## 4. Figura 13: Ranking Departamental de Tenencia de Computador

In [ ]:
set_style()

fig, ax = plt.subplots(figsize=(11, 13))

df_sorted = df_dpto.sort_values("pct_computador", ascending=True).reset_index(drop=True)
bar_colors = [REGION_COLORS.get(r, "#333333") for r in df_sorted["region"]]

bars = ax.barh(df_sorted["departamento"], df_sorted["pct_computador"],
 color=bar_colors, alpha=0.85, edgecolor="white", height=0.7)

nat_mean = (df_h["tiene_computador"] * df_h["fex_c"]).sum() / df_h["fex_c"].sum() * 100
ax.axvline(nat_mean, color="#d62728", linestyle="--", linewidth=1.5, zorder=4)

for bar, val in zip(bars, df_sorted["pct_computador"]):
 ax.text(val + 0.8, bar.get_y() + bar.get_height() / 2, f"{val:.1f}%",
 va="center", fontsize=8.5, color="#222222", weight="bold")

ax.set_xlim(0, 80)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title(
 "Tenencia de Computador en el Hogar por Departamento (33 Dptos.)\n"
 "Estimaciones Ponderadas con Factor de Expansión FEX_C — ECV 2024",
 fontsize=13, weight="bold", pad=15)
ax.set_xlabel("Porcentaje de Hogares con Computador o Portátil (%)", fontsize=10, labelpad=12)

legend_elements = [Patch(facecolor=c, label=r) for r, c in REGION_COLORS.items()]
legend_elements.append(plt.Line2D([0],[0], color="#d62728", linestyle="--",
 linewidth=1.5, label=f"Nacional: {nat_mean:.1f}%"))
ax.legend(handles=legend_elements, loc="lower right", framealpha=0.9,
 fontsize=8.5, title="Macro-Región DANE")

plt.tight_layout()
annotate_source(fig, "Fuente: DANE — ECV 2024. Ponderación con FEX_C. Cálculos propios.", x=0.04, y=-0.015)
save_fig(fig, "fig13_ranking_departamental_computador")
plt.show()

## 5. Figura 14: Capital Humano vs. Uso de Internet por Departamento

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 6.8))

x = df_dpto["pct_educ_superior_persona"]
y = df_dpto["pct_uso_internet_persona"]
sizes = df_dpto["hogares_expandidos"] / 3200
bubble_colors = [REGION_COLORS.get(r, "#555555") for r in df_dpto["region"]]

ax.scatter(x, y, s=sizes, c=bubble_colors, alpha=0.72,
 edgecolors="#222222", linewidth=0.8, zorder=3)

# Regresión lineal
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
x_vals = np.linspace(x.min() * 0.9, x.max() * 1.05, 100)
ax.plot(x_vals, intercept + slope * x_vals,
 color="#333333", linestyle="--", linewidth=1.5, zorder=2)

custom_offsets = {
 "Bogotá, D.C.": (0.6, -1.0),
 "Vaupés": (0.8, -3.2),
 "Vichada": (0.6, -2.8),
 "Guainía": (0.6, -2.5),
 "Chocó": (0.8, -2.8),
 "La Guajira": (0.8, -2.2),
 "Casanare": (-3.8, 1.2),
 "Antioquia": (0.8, 1.5),
 "Quindío": (-1.0, 3.2),
 "Atlántico": (0.8, -2.5),
}

for _, row in df_dpto.iterrows():
 name = row["departamento"]
 if name in custom_offsets:
 dx, dy = custom_offsets[name]
 ax.annotate(name,
 xy=(row["pct_educ_superior_persona"], row["pct_uso_internet_persona"]),
 xytext=(row["pct_educ_superior_persona"] + dx,
 row["pct_uso_internet_persona"] + dy),
 fontsize=8.5, weight="bold", color="#111111",
 arrowprops=dict(arrowstyle="->", color="#555555",
 lw=0.7, shrinkA=3, shrinkB=3))

ax.set_xlim(2, 40)
ax.set_ylim(12, 102)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title(
 "Heterogeneidad Territorial: Capital Humano vs. Uso Individual de Internet\n"
 f"Correlación Departamental (n=33), R²={r_value**2:.3f}, p<0.001 — ECV 2024",
 fontsize=12, weight="bold", pad=12)
ax.set_xlabel("Población con Educación Superior (%)", fontsize=10, labelpad=12)
ax.set_ylabel("Población que Usa Internet (%)", fontsize=10, labelpad=10)

legend_handles = [Patch(facecolor=c, label=r) for r, c in REGION_COLORS.items()]
legend_handles.append(plt.Line2D([0],[0], color="#333333", linestyle="--",
 label=f"R² = {r_value**2:.3f} (p < 0.001)"))
ax.legend(handles=legend_handles, loc="lower right", framealpha=0.92,
 fontsize=8.5, title="Macro-Región (Tamaño=Hogares)")

plt.tight_layout()
annotate_source(fig,
 "Fuente: DANE — ECV 2024. Ponderación FEX_C. Tamaño de burbuja proporcional a hogares expandidos.",
 x=0.05, y=-0.02)
save_fig(fig, "fig14_dispersion_territorial_educacion_vs_uso")
plt.show()

print(f"\nCorrelación de Pearson: r = {r_value:.3f}, R² = {r_value**2:.3f}, p = {p_value:.4f}")

## 6. Figura 15: Brecha Estructural Fijo vs. Móvil por Macro-Región

In [ ]:
df_reg = compute_regional_indicators(df_h, df_p)

fig, ax = plt.subplots(figsize=(10.5, 6))

df_reg_plot = df_reg.set_index("region")[["pct_computador", "pct_internet_fijo", "pct_internet_movil"]]
df_reg_plot.columns = ["Tiene Computador", "Internet Fijo", "Internet Móvil"]

df_reg_plot.plot(kind="bar", ax=ax, color=["#1f77b4", "#8c564b", "#17becf"],
 edgecolor="white", width=0.72)

ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title(
 "Brecha Estructural de Conectividad por Macro-Región (ECV 2024)\n"
 "Comparación Ponderada: Computador, Banda Ancha Fija e Internet Móvil",
 fontsize=12, weight="bold", pad=15)
ax.set_xlabel("Macro-Región Geográfica", fontsize=10, labelpad=12)
ax.set_ylabel("Penetración en Hogares (%)", fontsize=10, labelpad=10)
ax.set_ylim(0, 115)
ax.legend(title="Tecnología", loc="upper center", bbox_to_anchor=(0.5, 1.0),
 ncol=3, framealpha=0.95, fontsize=9)
plt.xticks(rotation=0)

for p in ax.patches:
 h = p.get_height()
 if h > 0:
 ax.annotate(f"{h:.1f}%", (p.get_x() + p.get_width() / 2., h + 1.8),
 ha="center", va="bottom", fontsize=8, weight="bold")

plt.tight_layout()
annotate_source(fig, "Fuente: DANE — ECV 2024. Ponderación con factor FEX_C.", x=0.05, y=-0.02)
save_fig(fig, "fig15_brecha_regional_fijo_movil_computador")
plt.show()

## 7. Tabla Comparativa Macro-Regional

In [ ]:
df_reg = compute_regional_indicators(df_h, df_p)

print("\nINDICADORES DIGITALES POR MACRO-REGIÓN — ECV 2024 (Estimaciones Ponderadas FEX_C)")
print("="*110)
cols_reg = ["region", "hogares_expandidos", "pct_computador", "pct_internet_fijo",
 "pct_internet_movil", "pct_alta_dificultad", "pct_urbano",
 "pct_uso_internet_persona", "pct_educ_superior_persona"]

df_reg_display = df_reg[cols_reg].copy()
df_reg_display.columns = [
 "Región", "Hogares (exp.)", "% Computador", "% Int. Fijo",
 "% Int. Móvil", "% Alta Dif. Eco.", "% Urbano",
 "% Usa Internet", "% Educ. Superior"
]
display(df_reg_display.to_string(index=False))

print("\n[Nota] Ponderación con FEX_C — DANE ECV 2024. Cálculos propios.")

## 8. Matriz de Correlación Departamental

In [ ]:
corr_vars = [
 "pct_computador", "pct_internet_total", "pct_internet_fijo",
 "pct_alta_dificultad", "pct_pobreza_multidimensional",
 "pct_uso_internet_persona", "pct_educ_superior_persona"
]

df_corr = df_dpto[corr_vars].copy()
df_corr.columns = [
 "Computador", "Internet Total", "Internet Fijo",
 "Alta Dif. Eco.", "Pobreza Multi.",
 "Usa Internet", "Educ. Superior"
]

corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
 cmap="RdBu_r", center=0, vmin=-1, vmax=1,
 square=True, linewidths=0.5, ax=ax,
 cbar_kws={"shrink": 0.7})

ax.set_title(
 "Matriz de Correlación de Pearson — Indicadores Departamentales (n=33)\n"
 "ECV 2024 DANE — Estimaciones Ponderadas con FEX_C",
 fontsize=11, weight="bold", pad=12)
plt.xticks(rotation=30, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
annotate_source(ax, x=0.02, y=-0.08)
save_fig(fig, "fig16_correlacion_departamental")
plt.show()

print("\nCorrelaciones clave con Tenencia de Computador:")
print(corr_matrix["Computador"].sort_values(ascending=False).to_string())

## 9. Síntesis del Análisis Territorial

### Hallazgos Principales

#### 9.1 Brecha Inter-Departamental Muy Pronunciada


#### 9.2 La Paradoja de la Conectividad Móvil


#### 9.3 Capital Humano como Predictor Territorial


#### 9.4 Resultados Inesperados de la Región Orinoquía


#### 9.5 Territorialización de la Exclusión Digital

- Alta proporción rural o semi-rural.
- Baja presencia de economía formal.
- Infraestructura de red escasa.
- Indicadores de pobreza multidimensional más elevados.